In [24]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re

In [25]:
URL = "https://sandbox.oxylabs.io/products"
session = requests.Session()
headers = {
    "User-Agent": "Mozilla/5.0"
}
session.headers.update(headers)

In [ ]:
# part 7
def request_catering(url, params=None):
    for i in range(5):
        try:
            if params:
                response = session.get(url, params=params, timeout=10, allow_redirects=True)
            else:
                response = session.get(url, timeout=10, allow_redirects=True)

            if response.status_code == 200:
                return response
            else:
                print("Request failed:", response.status_code,"Attempt:",i + 1, "for", url)
        except requests.RequestException as e:
            print("Error:", e,"Attempt:",i + 1)
            
        time.sleep(2)
    print("All retry attempts failed for:", url)
    return None

Reasoning:
 to make it resilient, we do the retry strategy and do it for a certain time of attempts to get if it gets OK response or not and if it still fails then the it jumps outside without crashing, and previous scraped products remain in the  "products" list

In [48]:

test_session = requests.Session()
test_session.headers.update(headers)

test_session.get(URL)  # establish session on page 1
session_response = test_session.get(URL, params={"page": 2})
fresh_response=requests.get(f"{URL}?page=2", headers=headers)
print("session response ->",session_response.url)
print("fresh response ->",fresh_response.url)
    

session response -> https://sandbox.oxylabs.io/products?page=2
fresh response -> https://sandbox.oxylabs.io/products?page=2


reasoning:


In [49]:
def productLinks(product_links):
    page_products=[]
    for link in product_links:

        href = link.get("href")
        if not href:
            continue

        if href.startswith("/products/") and not href.startswith("/products/category"):
            product_url = requests.compat.urljoin(current_url, href)

            # handling duplicates
            if product_url not in visited_products:
                visited_products.add(product_url)
                page_products.append(product_url)

    return page_products

In [50]:
def detail_page(page_products):

    for product_url in page_products:

        product_response = request_catering(product_url)

        if product_response is None:
            print("Could not load:", product_url)
            continue

        product_soup = BeautifulSoup(
            product_response.content,
            "html.parser"
        )

        # Product name
        name_tag = product_soup.find("h2", class_="title")

        if name_tag:
            name = name_tag.get_text(strip=True)
        else:
            name = ""


        # Price
        price_tag = product_soup.find(
            "div",
            class_="price"
        )

        if price_tag:
            price = price_tag.get_text(strip=True)
        else:
            price = ""


        # Stock status
        stock_tag = product_soup.find(
            "p",
            class_="availability"
        )

        if stock_tag:
            stock_status = stock_tag.get_text(strip=True).lower()
        else:
            stock_status = ""


        # Description
        description_tag = product_soup.find(
            "p",
            class_="description"
        )

        if description_tag:
            description = description_tag.get_text(
                " ",
                strip=True
            )
        else:
            description = ""


        products.append({
            "Product name": name,
            "Price": price,
            "Stock status": stock_status,
            "Description": description,
            "Detail-page URL": product_url,
            "Listing page number": page_number
        })

        print("  ", name)

In [ ]:
products = []

page_number = 1
visited_pages = set()
visited_products = set()

current_url = URL

while current_url:

    
    if current_url in visited_pages:
        break

    visited_pages.add(current_url)

    print("Scraping listing page:", page_number)

    response = request_catering(current_url)

    if response is None:
        print("Could not load page. Stopping.")
        break

    soup = BeautifulSoup(response.content, "html.parser")

    # Find all product links
    product_links = soup.find_all("a")

    page_products = []

    page_products=productLinks(product_links)
    

    print("Products found:", len(page_products))

    detail_page(page_products)
    
    
    
    # Find Forward button
    forward_link = None

    for link in soup.find_all("a"):

        text = link.get_text(" ", strip=True).lower()

        if text == "forward":
            forward_link = link
            break


    
    if forward_link is None:
        print("Forward button not found.")
        break


    # Check if Forward button is disabled
    if forward_link.get("aria-disabled") == "true":
        print("Forward button is disabled.")
        print("Final page reached:", page_number)
        break


    href = forward_link.get("href")

    next_link = requests.compat.urljoin(
        current_url,
        href
    )


    current_url = next_link
    page_number += 1

    time.sleep(0.5)

Scraping listing page: 1
Products found: 32
   The Legend of Zelda: Ocarina of Time
   Super Mario Galaxy
   Super Mario Galaxy 2
   Metroid Prime
   Super Mario Odyssey
   Halo: Combat Evolved
   The House in Fata Morgana - Dreams of the Revenants Edition -
   NFL 2K1
   Uncharted 2: Among Thieves
   Tekken 3
   The Legend of Zelda: The Wind Waker
   Gran Turismo
   Metal Gear Solid 2: Sons of Liberty
   Grand Theft Auto Double Pack
   Baldur's Gate II: Shadows of Amn
   Tetris Effect: Connected
   The Legend of Zelda Collector's Edition
   Gran Turismo 3: A-Spec
   The Legend of Zelda: A Link to the Past
   The Legend of Zelda: Majora's Mask
   The Last of Us
   Persona 5 Royal
   The Last of Us Remastered
   The Legend of Zelda: Ocarina of Time 3D
   Chrono Cross
   Gears of War
   Sid Meier's Civilization II
   Halo 3
   Ninja Gaiden Black
   Super Mario Advance 4: Super Mario Bros. 3
   Jet Grind Radio
   Grim Fandango
Scraping listing page: 2
Products found: 32
   Resident Evil C

KeyboardInterrupt: 

In [ ]:
df = pd.DataFrame(products)

print(df)

print("Total products:", len(df))
print("Total listing pages:", len(visited_pages))

                              Product name    Price  Stock status  \
0     The Legend of Zelda: Ocarina of Time  91,99 €      in stock   
1                       Super Mario Galaxy  91,99 €  out of stock   
2                     Super Mario Galaxy 2  91,99 €      in stock   
3                            Metroid Prime  89,99 €  out of stock   
4                      Super Mario Odyssey  89,99 €      in stock   
...                                    ...      ...           ...   
2995                              Crashday  76,99 €  out of stock   
2996                               The Con  72,99 €      in stock   
2997                           Van Helsing  76,99 €  out of stock   
2998                             Rogue Ops  63,99 €      in stock   
2999                       Black & Bruised  63,99 €  out of stock   

                                            Description  \
0     As a young boy, Link is tricked by Ganondorf, ...   
1     [Metacritic's 2007 Wii Game of the Year] The u.

In [ ]:
df = df.drop_duplicates(
    subset=["Detail-page URL"]
)

df = df.reset_index(drop=True)

print("Products after duplicate removal:", len(df))
print(df["Detail-page URL"].duplicated().sum())

Products after duplicate removal: 3000
0


In [ ]:
filename = "23L_0570_versionA_static_products.csv"

df.to_csv(filename, index=False)

print("CSV saved:", filename)

CSV saved: 23L_0570_versionA_static_products.csv


Question 1 – Scraping Methodology

How you identified the relevant elements/records on each page ?
answer:
by inspecting each product card and finding what hierarchy or pattern each card has similar

How you navigated across pages
Answer:
there is a forward button for pagination,

How your program decided that scraping was complete
Answer: when reaches the last page that forward button disables

Any challenges you ran into and how you resolved them.
Answer: while implementing the retry strategy for the failed requests.

How you verified that your scraper was actually collecting correct, complete data (spot checks,counts, etc.)?
Answer: ok so i checked each page and count of product on each page and then multiplied them and also dod spot checking that if that certain product is in that certain page. and at the start some of pages were scraped and some were not so had to work on them.

Question 1 – Validation Statistics
Total listing pages processed: 94
Total product URLs collected: 3000
Total records extracted with required fields: 3000
Duplicate or empty records removed/skipped: none
Failed requests retried and recovered: none